# lltGPT, Assemble a Modern GPT

LLM Training Lab, module 4. This notebook runs every step of the lab page on a CPU. No GPU is needed. Our recorded runs are in `courses/llm-training/runs/lltGPT/`.

In [ ]:
# Runtime check. Run this first. It prints the fix for each problem it finds.
import platform, shutil, sys
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("FIX: transformers 5.22.1 needs Python 3.11 or later. On Colab, keep the default runtime.")
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f"RAM {ram:.1f} GB")
    if ram < 4:
        print("FIX: this lab peaked at 2.87 GB of RAM in our run. Use a machine or runtime with 4 GB or more.")
except ImportError:
    print("psutil not installed, RAM not checked")
free = shutil.disk_usage(".").free / 1e9
print(f"Free disk {free:.1f} GB")
if free < 2:
    print("FIX: the GPT-2 download needs 0.55 GB plus cache space. Free some disk space.")
try:
    import torch
    print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except ImportError:
    print("FIX: install PyTorch, pip install torch")
try:
    import numpy
    print("numpy", numpy.__version__, "found | our CPU runs used 2.5.4")
except ImportError:
    print("FIX: install NumPy, pip install numpy")
print("This lab runs on a CPU. No GPU is needed.")

Install the pinned versions. PyTorch and NumPy are already installed on Colab and Kaggle, so keep the image's versions. Our CPU figures were measured on torch 2.15.0 and numpy 2.5.4.

In [ ]:
%pip install -q "transformers==5.22.1" "tokenizers==0.23.3" "huggingface_hub==2.4.0"

## Step 1. Check the RoPE property

The same NumPy code the page runs in your browser.

In [ ]:
import numpy as np

def rope(x, pos, base=10000.0):
    # Rotate each pair (x[2i], x[2i+1]) by the angle pos * base**(-2i/d)
    d = x.shape[-1]
    ang = pos * base ** (-np.arange(0, d, 2) / d)
    c, s = np.cos(ang), np.sin(ang)
    out = np.empty_like(x)
    out[0::2] = x[0::2] * c - x[1::2] * s
    out[1::2] = x[0::2] * s + x[1::2] * c
    return out

rng = np.random.default_rng(0)
q, k = rng.normal(size=64), rng.normal(size=64)

# A query at position m and a key at position n, always 3 apart
for m in [3, 10, 100, 1000]:
    print(f"RoPE         m = {m:>4}  n = {m - 3:>3}  score {rope(q, m) @ rope(k, m - 3):.6f}")
scores = [rope(q, m) @ rope(k, m - 3) for m in range(3, 2000)]
print(f"RoPE, 1,997 pairs with gap 3, max - min = {max(scores) - min(scores):.1e}")
print(f"RoPE, gap 5 instead   score {rope(q, 5) @ rope(k, 0):.6f}")
print(f"length of q {np.linalg.norm(q):.6f}, after rotating to position 1000 {np.linalg.norm(rope(q, 1000)):.6f}")

# The GPT-2 way: add a position vector to q and k instead of rotating them
P = rng.normal(size=(2000, 64))
for m in [3, 10, 100, 1000]:
    print(f"added vector m = {m:>4}  n = {m - 3:>3}  score {(q + P[m]) @ (k + P[m - 3]):.6f}")

## Step 2. Write the config and the two norms

In [ ]:
import math
from dataclasses import dataclass, replace

import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Config:
    vocab: int = 50257
    ctx: int = 1024
    d: int = 768
    layers: int = 12
    heads: int = 12
    norm: str = "rmsnorm"   # "rmsnorm" or "layernorm"
    mlp: str = "swiglu"     # "swiglu" or "gelu"
    pos: str = "rope"       # "rope" or "learned"
    prenorm: bool = True
    bias: bool = False
    eps: float = 1e-5

GPT2 = Config(norm="layernorm", mlp="gelu", pos="learned", bias=True)
MODERN = Config()

class RMSNorm(nn.Module):
    def __init__(self, d, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

def make_norm(cfg):
    if cfg.norm == "rmsnorm":
        return RMSNorm(cfg.d, cfg.eps)
    return nn.LayerNorm(cfg.d, eps=cfg.eps, bias=cfg.bias)

torch.manual_seed(0)
x = torch.randn(2, 5, 768) * 3 + 1   # mean 1, std 3
for name, norm in [("LayerNorm", make_norm(GPT2)), ("RMSNorm", make_norm(MODERN))]:
    y = norm(x)
    n = sum(p.numel() for p in norm.parameters())
    print(f"{name:9}  mean {y.mean(-1).mean():.4f}  rms {y.pow(2).mean(-1).sqrt().mean():.4f}  parameters {n}")

## Step 3. Write attention, the two MLPs and the block

In [ ]:
def rope_tables(T, hd, offset=0, base=10000.0, device=None):
    inv = base ** (-torch.arange(0, hd, 2, device=device).float() / hd)
    ang = torch.outer(torch.arange(offset, offset + T, device=device).float(), inv)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):
    # x is (batch, heads, T, head_dim). Rotate the pairs (x[2i], x[2i+1]) as in step 1.
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.rope = cfg.heads, cfg.pos == "rope"
        self.qkv = nn.Linear(cfg.d, 3 * cfg.d, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d, cfg.d, bias=cfg.bias)

    def forward(self, x, cos=None, sin=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.heads, C // self.heads).transpose(1, 2) for t in (q, k, v))
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)   # queries and keys, never values
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

def swiglu_hidden(d):
    # 8d/3 keeps the parameter count of a 4d GELU MLP; round up to a multiple of 64
    return 64 * math.ceil(8 * d / 3 / 64)

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.swiglu = cfg.mlp == "swiglu"
        hidden = swiglu_hidden(cfg.d) if self.swiglu else 4 * cfg.d
        self.gate = nn.Linear(cfg.d, hidden, bias=cfg.bias) if self.swiglu else None
        self.up = nn.Linear(cfg.d, hidden, bias=cfg.bias)
        self.down = nn.Linear(hidden, cfg.d, bias=cfg.bias)

    def forward(self, x):
        if self.swiglu:
            return self.down(F.silu(self.gate(x)) * self.up(x))
        return self.down(F.gelu(self.up(x), approximate="tanh"))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.prenorm = cfg.prenorm
        self.n1, self.n2 = make_norm(cfg), make_norm(cfg)
        self.attn, self.mlp = Attention(cfg), MLP(cfg)

    def forward(self, x, cos=None, sin=None):
        if self.prenorm:
            x = x + self.attn(self.n1(x), cos, sin)
            return x + self.mlp(self.n2(x))
        x = self.n1(x + self.attn(x, cos, sin))
        return self.n2(x + self.mlp(x))

for d in [192, 320, 768, 4096]:
    h = swiglu_hidden(d)
    print(f"d = {d:>4}  8d/3 = {8 * d / 3:7.1f}  hidden {h:>5}  MLP weights {3 * d * h:>11,}  GELU 4d MLP {8 * d * d:>11,}")

## Step 4. Assemble the GPT, tie the head and scale the init

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab, cfg.d)
        self.pos = nn.Embedding(cfg.ctx, cfg.d) if cfg.pos == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])
        self.norm = make_norm(cfg)
        self.head = nn.Linear(cfg.d, cfg.vocab, bias=False)
        self.head.weight = self.tok.weight   # tie first, so the init below sets the shared tensor once
        for name, p in self.named_parameters():
            if name.endswith(("proj.weight", "down.weight")):
                # the two layers per block that write into the residual stream
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.layers))
            elif p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02)
            elif name.endswith("bias"):
                nn.init.zeros_(p)

    def forward(self, idx, offset=0):
        B, T = idx.shape
        x = self.tok(idx)
        cos = sin = None
        if self.pos is None:
            cos, sin = rope_tables(T, self.cfg.d // self.cfg.heads, offset, device=idx.device)
        else:
            x = x + self.pos(torch.arange(offset, offset + T, device=idx.device))
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.head(self.norm(x))

# The model our TinyStories pretraining run uses (lltTinyStories)
TINY = replace(MODERN, vocab=4096, ctx=512, d=320, layers=8, heads=8)
torch.manual_seed(0)
model = GPT(TINY)
print("head is the embedding tensor:", model.head.weight is model.tok.weight)
print(f"init std  qkv {model.blocks[0].attn.qkv.weight.std():.4f}  proj {model.blocks[0].attn.proj.weight.std():.4f}"
      f"  down {model.blocks[0].mlp.down.weight.std():.4f}  target 0.02/sqrt(16) = {0.02 / math.sqrt(16):.4f}")
idx, targets = torch.randint(0, TINY.vocab, (2, 4, 128))
with torch.no_grad():
    loss = F.cross_entropy(model(idx).flatten(0, 1), targets.flatten())
print(f"step-0 loss {loss:.3f}  ln(V) = {math.log(TINY.vocab):.3f}")

## Step 5. Test shapes and causal leakage

In [ ]:
SMALL = dict(vocab=1000, ctx=64, d=64, layers=2, heads=4)

def leak_test(model, T=16, t=7):
    # Change the token at position t + 1 and measure how far each position's logits move
    idx = torch.randint(0, model.cfg.vocab, (1, T))
    changed = idx.clone()
    changed[0, t + 1] = (idx[0, t + 1] + 1) % model.cfg.vocab
    with torch.no_grad():
        moved = (model(idx) - model(changed)).abs().amax(dim=-1)[0]
    return moved[: t + 1].max().item(), moved[t + 1].item()

torch.manual_seed(0)
for name, base in [("gpt2", GPT2), ("modern", MODERN)]:
    model = GPT(replace(base, **SMALL)).eval()
    idx = torch.randint(0, 1000, (2, 16))
    out = model(idx)
    before, at = leak_test(model)
    with torch.no_grad():
        shifted = (model(idx) - model(idx, offset=40)).abs().max().item()
    print(f"{name:6}  logits {tuple(out.shape)}  positions 0-7 moved {before:.1e}  position 8 moved {at:.2f}"
          f"  every position shifted by 40 moved {shifted:.1e}")
    assert out.shape == (2, 16, 1000)
    assert before == 0.0, "a later token changed an earlier output; check the causal mask"
print("shape and leakage tests passed")

## Step 6. Count the parameters and the FLOPs per token

In [ ]:
def numel(model):
    return sum(p.numel() for p in model.parameters())   # a tied tensor counts once

def approx_params(cfg):
    return 12 * cfg.layers * cfg.d ** 2 + cfg.vocab * cfg.d

TOGGLE = replace(MODERN, vocab=2048, ctx=128, d=192, layers=4, heads=6)   # the step 8 experiment
for name, cfg in [("GPT-2 124M", GPT2), ("modern, GPT-2 size", MODERN), ("TinyStories model", TINY), ("step 8 model", TOGGLE),
                  ("step 8, 50k vocab", replace(TOGGLE, vocab=50257))]:
    n, a, emb = numel(GPT(cfg)), approx_params(cfg), cfg.vocab * cfg.d
    print(f"{name:18}  sum(numel) {n:>11,}  12Ld^2 + Vd {a:>11,}  gap {(n - a) / n:+.2%}  embedding {emb / n:.0%}")

In [ ]:
from torch.nn.attention import SDPBackend, sdpa_kernel
from torch.utils.flop_counter import FlopCounterMode

T, cfg = 1024, GPT2
model = GPT(cfg).eval()
# The counter sees attention only through the plain-math backend, so force it for this count
with FlopCounterMode(display=False) as counter, torch.no_grad(), sdpa_kernel(SDPBackend.MATH):
    model(torch.zeros(1, T, dtype=torch.long))
counted = counter.get_total_flops() / T
weights = 12 * cfg.layers * cfg.d ** 2 + cfg.vocab * cfg.d   # every weight in a matmul, the head included
attention = 4 * cfg.layers * T * cfg.d                        # scores Q K^T, then the weighted sum of V
print(f"forward FLOPs per token at T = {T}  counted {counted / 1e6:.1f}M  formula 2 x weights + attention = {(2 * weights + attention) / 1e6:.1f}M")
print(f"attention share {attention / (2 * weights + attention):.1%}  training is about 3x forward, {3 * (2 * weights + attention) / 1e6:.0f}M per token")

## Step 7. Load GPT-2 124M and match Hugging Face's logits

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2", dtype=torch.float32).eval()
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")

# HF GPT-2 keeps these four layers as Conv1D, weight shape (in, out). nn.Linear wants (out, in).
CONV1D = {"attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"}
RENAME = {"ln_1": "n1", "ln_2": "n2", "attn.c_attn": "attn.qkv", "attn.c_proj": "attn.proj",
          "mlp.c_fc": "mlp.up", "mlp.c_proj": "mlp.down"}

def gpt2_state_dict(sd):
    out = {"tok.weight": sd["transformer.wte.weight"], "pos.weight": sd["transformer.wpe.weight"],
           "norm.weight": sd["transformer.ln_f.weight"], "norm.bias": sd["transformer.ln_f.bias"]}
    out["head.weight"] = out["tok.weight"]
    for key, w in sd.items():
        if key.startswith("transformer.h."):
            layer, rest = key[len("transformer.h."):].split(".", 1)
            module, kind = rest.rsplit(".", 1)
            if module in CONV1D and kind == "weight":
                w = w.t()
            out[f"blocks.{layer}.{RENAME[module]}.{kind}"] = w
    return out

ours = GPT(GPT2).eval()
ours.load_state_dict(gpt2_state_dict(hf.state_dict()))   # strict, so a missing or extra key raises
text = "Once upon a time, a small robot lived in a quiet town. Every morning it walked to the park and counted the birds."
ids = tok(text, return_tensors="pt").input_ids
with torch.no_grad():
    ref, out = hf(ids).logits, ours(ids)
diff = (out - ref).abs().max().item()
print(f"{ids.shape[1]} tokens  largest |logit| {ref.abs().max():.1f}  max abs difference {diff:.2e}"
      f"  same top token everywhere {bool((out.argmax(-1) == ref.argmax(-1)).all())}")
print(f"loss  ours {F.cross_entropy(out[0, :-1], ids[0, 1:]):.4f}  HF {F.cross_entropy(ref[0, :-1], ids[0, 1:]):.4f}")
assert torch.allclose(out, ref, rtol=1e-5, atol=1e-4), "logits differ; check the Conv1D transposes and the GELU variant"

prompt = tok("Once upon a time, a small robot", return_tensors="pt").input_ids
with torch.no_grad():
    for _ in range(12):
        prompt = torch.cat([prompt, ours(prompt)[:, -1:].argmax(-1)], dim=1)
print(repr(tok.decode(prompt[0])))

## Step 8. Toggle each modern component

Predict first. Which swap back to GPT-2's choice raises the validation loss most after 600 steps? Each run takes several minutes on a laptop CPU.

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

path = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset")
stories = [s.strip() for s in open(path, encoding="utf-8").read().split("<|endoftext|>") if s.strip()]
split = int(len(stories) * 0.95)   # 95% to train on, the last 5% held out
bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.decoder = decoders.ByteLevel()
bpe.train_from_iterator(stories[:split], trainers.BpeTrainer(vocab_size=TOGGLE.vocab, special_tokens=["<|eot|>"],
                                                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))

def encode(texts):
    ids = []
    for enc in bpe.encode_batch(texts):
        ids += enc.ids + [bpe.token_to_id("<|eot|>")]
    return torch.tensor(ids)

train_ids, val_ids = encode(stories[:split]), encode(stories[split:])
VAL_X = val_ids[: 256 * 128].view(256, 128)          # 256 fixed windows of 128 tokens
VAL_Y = val_ids[1 : 256 * 128 + 1].view(256, 128)
print(f"{len(stories):,} stories  train tokens {len(train_ids):,}  val tokens {len(val_ids):,}")

@torch.no_grad()
def val_loss(model):
    model.eval()
    loss = sum(F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item() for x, y in zip(VAL_X.split(64), VAL_Y.split(64))) / 4
    model.train()
    return loss

def train(cfg, steps=600, seed=0, lr=3e-3, batch=32, warmup=50, every=100):
    torch.manual_seed(seed)
    model, gen = GPT(cfg), torch.Generator().manual_seed(seed)
    groups = [{"params": [p for p in model.parameters() if p.dim() >= 2], "weight_decay": 0.1},
              {"params": [p for p in model.parameters() if p.dim() < 2], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95))
    curve = [(0, val_loss(model))]
    for step in range(steps):
        frac = (step + 1) / warmup if step < warmup else 0.1 + 0.45 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))
        for g in opt.param_groups:
            g["lr"] = lr * frac
        starts = torch.randint(0, len(train_ids) - cfg.ctx - 1, (batch,), generator=gen)
        x = torch.stack([train_ids[s : s + cfg.ctx] for s in starts])
        y = torch.stack([train_ids[s + 1 : s + cfg.ctx + 1] for s in starts])
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if (step + 1) % every == 0:
            curve.append((step + 1, val_loss(model)))
    return model, curve

In [ ]:
VARIANTS = {
    "modern": TOGGLE,
    "learned positions": replace(TOGGLE, pos="learned"),
    "GELU MLP": replace(TOGGLE, mlp="gelu"),
    "LayerNorm": replace(TOGGLE, norm="layernorm"),
    "post-norm": replace(TOGGLE, prenorm=False),
    "gpt2 config": replace(TOGGLE, norm="layernorm", mlp="gelu", pos="learned", bias=True),
}
SEEDS = [0]   # our table used seeds 0 and 1
for name, cfg in VARIANTS.items():
    for seed in SEEDS:
        model, curve = train(cfg, seed=seed)
        print(f"{name:18} seed {seed}  parameters {numel(model):,}  val loss at step 600 {curve[-1][1]:.4f}")

In [ ]:
# Peak RAM of this notebook so far
import psutil, sys
mi = psutil.Process().memory_info()
peak = getattr(mi, "peak_wset", None)  # Windows
if peak is None:
    import resource  # Linux reports ru_maxrss in KB, macOS in bytes
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak = r * 1024 if sys.platform.startswith("linux") else r
print(f"Peak RAM {peak / 1e9:.2f} GB")